# Central Perspective Imaging Model — Programming Examples

This notebook follows along with the *Programming Example* slides on the Central Perspective Imaging Model of **Lecture 26: Introduction to Computer Vision**. We use the `machinevisiontoolbox` package's `CentralCamera` class to:

1. Project a 3D world point onto the image plane and onto pixel coordinates.
2. Inspect the camera's intrinsic matrix $K$ and camera matrix $C$.
3. Check the **visibility** of a set of 3D points.
4. Project a **group of points** (a grid) and a **3D primitive** (a cube), and see how translating the camera affects the projected image.

## Environment Setup

Detect whether the notebook is running on Google Colab (installing `matplotlib` and `machinevision-toolbox-python` if so), then import everything this notebook needs: `numpy`, `matplotlib`, the Machine Vision Toolbox (`machinevisiontoolbox`), and the Spatial Math Toolbox (`spatialmath`). Run this cell first.

In [ ]:
try:
    import google.colab
    print('Running on CoLab')
    !pip install matplotlib
    !pip install machinevision-toolbox-python
    COLAB = True
except ModuleNotFoundError:
    COLAB = False

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
np.set_printoptions(
    linewidth=120, formatter={
        'float': lambda x: f"{0:8.4g}" if abs(x) < 1e-10 else f"{x:8.4g}"})
np.random.seed(0)

from machinevisiontoolbox import CentralCamera, mkgrid, mkcube
from spatialmath import SE2, SE3, Twist2


## 1. Project a World Point onto the Image Plane

**Problem:** Project a world point $(3, 0.4, 5)^T$ m using a camera with a 15 mm lens.

We create a `CentralCamera` object with a 15 mm focal length and project a 3D world point $(X, Y, Z)$ onto the image plane. The result is expressed in metres, relative to the principal point. We then move the camera 0.5 m to the left along the $x$-axis and project the same point again to see how the image point shifts.

In [ ]:
# Create a camera object with 15mm lens
camera = CentralCamera(f=0.015)

# Define a 3D world point (X, Y, Z) in meters
P = [0.3, 0.4, 5.0]

# Project the point onto the image plane
p = camera.project_point(P)            # Output: [[0.0009], [0.0012]] -> (1.5mm, 2.0mm)
p

In [ ]:
# Move the camera 0.5m to the left and project again
p_moved = camera.project_point(P, pose=SE3.Tx(-0.5))
p_moved             # Output: [[0.0024], [0.0012]] -> The image point moves to the right

## 2. Project to Pixel Coordinates

**Problem:** Define a camera with:
- 15 mm focal length ($f$)
- 10 µm square photosites ($w = h = \rho$)
- $1280 \times 1024$ resolution (`imagesize`)
- Principal point at $(640, 512)$ (`pp` $\to (u_0, v_0)$)

and project the same 3D point `P` directly to **pixel** coordinates `p_pixel`. We then inspect the resulting **camera intrinsic matrix** $K$ and the full **camera matrix** $C$.

In [ ]:
camera  = CentralCamera(f=0.015, rho=10e-6,
                        imagesize=[1280, 1024], pp=[640, 512])
p_pixel = camera.project_point(P)
p_pixel         # Output: [[730.0], [632.0]]

In [ ]:
# View the resulting Intrinsic Matrix K
camera.K
# Output:
# array([[    1500,        0,      640],
#        [       0,     1500,      512],
#        [       0,        0,        1]])

In [ ]:
# View the Camera Matrix C
camera.C()
# Output:
# array([[    1500,        0,      640,        0],
#        [       0,     1500,      512,        0],
#        [       0,        0,        1,        0]])

## 3. Visibility Checking of Points

A critical feature of digital sensing is **visibility checking**: determining whether a projected 3D point actually falls within the physical boundaries of the sensor ($W \times H$ pixels).

Here we project three 3D points at once and ask `project_point` to also return a `visible` flag for each — `True` if the projected pixel falls inside the image, `False` otherwise.

In [ ]:
# Visibility checking (say for 3 points)
P_points = np.column_stack([[0, 0, 10], [10, 10, 10], [0.3, 0.4, 5]])
p, visible = camera.project_point(P_points, visibility=True)
# p
# array([[     640,     2140,      730],
#        [     512,     2012,      632]])
# visible
# array([ True, False, True])
p, visible

## 4. Projecting a Group of Points (Grid)

Projecting a regular **grid of points** is a common way to visualize the geometric effects of perspective projection:

- In a **frontal** view, the grid appears regular and symmetric.
- In an **oblique** view (captured by changing the camera pose), the grid shape is distorted, and parallel edges in the world are projected as lines that converge toward a vanishing point.

We project a $3\times3$ grid `P_grid` of side length $0.2\ m$, centered at $(0, 0, 1)$, first from the camera's original pose, then again after translating the camera $-0.25\ m$ along the $x$-axis.

In [ ]:
P_grid = mkgrid(n=3, side=0.2, pose=SE3.Tz(1.0))

# Pixel Coordinates
P_pixel = camera.project_point(P_grid)
camera.plot_point(P_grid)

# translate camera along -x axis
T_camera = SE3.Trans(-0.25, 0, 0)
camera.plot_point(P_grid, pose=T_camera)

## 5. Projecting a 3D Primitive (Cube)

Beyond simple points, the toolbox can project complex **3D primitives** like cubes. We project the wireframe of a cube of side $0.2\ m$, positioned at $(0, 0, 1)$, first from the camera's original pose, then again after translating the camera $-0.25\ m$ along the $x$-axis.

In [ ]:
# Projecting a Cube wireframe of side 0.2m, position at (0, 0, 1)
X, Y, Z = mkcube(0.2, pose=SE3.Tz(1), edge=True)
camera.plot_wireframe(X, Y, Z)


In [ ]:
# translate camera along -x axis
T_camera = SE3.Trans(-0.25, 0, 0)
camera.plot_wireframe(X, Y, Z, pose=T_camera)


Translating the camera results in:
- The cube shifting in the **opposite** direction in the image.
- Edges/lines belonging to the **same plane** remaining parallel.
- Edges/lines in the direction of **depth** converging toward a vanishing point.

---
### Try it yourself
- Change the focal length `f`, photosite size `rho`, or `imagesize` when creating the `CentralCamera` and see how the intrinsic matrix `K` changes.
- Move the camera along a different axis (e.g. `SE3.Ty(...)` or `SE3.Tz(...)`) and observe how the grid/cube projection shifts.
- Add more points to `P_points` — some inside the sensor bounds, some outside — and check the `visible` flags.